In [ ]:
import pandas as pd
import os
import networkx as nx
import matplotlib.pyplot as plt
import math
from torch_geometric.data import Data
import itertools
import re
from collections import Counter
import gensim
import numpy as np
import scipy.sparse as sp
import pickle
import csv
import datetime
import json
import torch

In [ ]:
BASE_PATH = "Fakeddit"

### Loading the source posts

In [ ]:
df=pd.read_csv(os.path.join(BASE_PATH,"all_train.tsv"), sep='\t', header=0)
df2=pd.read_csv(os.path.join(BASE_PATH,"all_validate.tsv"), sep='\t', header=0)
df3=pd.read_csv(os.path.join(BASE_PATH,"all_test_public.tsv"), sep='\t', header=0)

In [ ]:
df.head()

In [ ]:
## Combining train,test and valid into one dataframe and clearing those don't have any text in it
combined_df=pd.DataFrame()
combined_df = pd.concat([df,df2,df3], ignore_index=True)
print(len(combined_df))
combined_df = combined_df.dropna(subset=["clean_title"])
print(len(combined_df))
combined_df = combined_df.reset_index(drop=True)

### Loading the Comments

In [ ]:
comments=pd.read_csv(os.path.join(BASE_PATH,"all_comments.tsv"), sep='\t', header=0)

In [ ]:
comments.head()

In [ ]:
len(comments)

### Mapping Source posts to comment chains

In [ ]:
top_post_ids=[i[3:] for i in comments["parent_id"] if str(i)!='nan' and i[0:3]=='t3_']
top_post_ids = list(set(top_post_ids))

In [ ]:
print(len(top_post_ids))

In [ ]:
mask = combined_df["id"].isin(top_post_ids)
all_top_posts = combined_df[mask]
print(len(all_top_posts))

In [ ]:
all_top_posts=all_top_posts[['id','created_utc','author','clean_title','2_way_label','num_comments']]

In [ ]:
labels_mapping = dict()
times_mapping = dict()
for i,j,k in zip(all_top_posts["id"],all_top_posts["2_way_label"],all_top_posts["created_utc"]):
    labels_mapping[i]=j
    dt_object = datetime.datetime.fromtimestamp(k)
    times_mapping[i]=dt_object.strftime("%Y-%m")
print(len(labels_mapping))

In [ ]:
uni=list(all_top_posts["id"].unique())
comm=comments[comments["submission_id"].isin(uni)]
comm.loc[:,'parent_id'] = comm['parent_id'].copy().str[3:]
grp=comm.groupby(["submission_id"])

In [ ]:
len(grp)

### Preparing Sentences and Users list

In [ ]:
## Mapping the id to content and id to author
content_id = dict()
id_to_user = dict()

for text,id_,user in zip(all_top_posts["clean_title"],all_top_posts["id"],all_top_posts["author"]):
    content_id[id_]=text
    id_to_user[id_]=user

for name, group in grp:
    for text,id_,user in zip(group["body"],group["id"],group["author"]):
        if str(text)!="nan": ## Ignoring 425 comments in total for fakeddit
            content_id[id_]=text
            id_to_user[id_]=user
            
print(len(content_id))

In [ ]:
sentences =[]
ids =[]
users = []
for i in content_id.keys():
    ids.append(i)
    sentences.append(content_id[i])
    users.append(id_to_user[i])
    

In [ ]:
print(len(id_to_user))

## overlap_ratio


In [ ]:
overlap_ratio=dict()

In [ ]:
with open("train_ids.pkl","rb") as f:
     train_ids = pickle.load(f)
with open("valid_ids.pkl","rb") as f:
     valid_ids = pickle.load(f)
with open("test_ids.pkl","rb") as f:
     test_ids = pickle.load(f)

In [ ]:
datetime.datetime.fromtimestamp(combined_df[combined_df["id"]==test_ids[-1]]["created_utc"].item()).strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Considering only the test split for graph construction
mask = all_top_posts["id"].isin(test_ids)
all_test_posts = all_top_posts[mask]

test_uni=list(all_test_posts["id"].unique())
# print(len(uni))
test_comm=comments[comments["submission_id"].isin(test_uni)]
test_comm.loc[:,'parent_id'] = test_comm['parent_id'].copy().str[3:]
test_grp=test_comm.groupby(["submission_id"])

In [ ]:
print(len(test_grp))

### Load User features and generate user representations

In [ ]:
# Load user embeddings and mapping 
with open("User_features.pkl","rb") as f:
    user_vectors = pickle.load(f)

with open("user_to_id.json","r") as f:
    user_to_id = json.load(f)  


In [ ]:
global_graph_users=set(user_to_id.keys())

In [ ]:
print(len(global_graph_users))

In [ ]:
set(user_to_id.keys())-set(users)

In [ ]:
t=0
cnt=0
for name, group in test_grp:
    total=1
    needed=0
    if id_to_user[name[0]] in global_graph_users:
        needed+=1
    for text,id_,user in zip(group["body"],group["id"],group["author"]):
        if str(text)!="nan": ## Ignoring 425 comments in total for fakeddit
            total+=1
            if user in global_graph_users:
                needed+=1
        # print(total,needed)
    # print(name[0])
    
    overlap_ratio[name[0]]=needed/total
    # print(needed,total)
    t+=1
    if(overlap_ratio[name[0]]<0.1):
        cnt+=1

In [ ]:
print(cnt)

In [ ]:
print(len(overlap_ratio))

In [ ]:
with open("overlap_ratio.json","w") as f:
    json.dump(overlap_ratio,f)

In [ ]:
user_embeds = dict()
users = set(users)
c1=0
c2=0

column_averages = np.mean(user_vectors, axis=0)
user_embeds["dummy"]=column_averages

for user in users:
    if user in user_to_id:
        user_embeds[user]=user_vectors[user_to_id[user]]
        c1+=1
    # elif str(user)!="nan" and user not in user_to_id:
    #     user_embeds[user]=np.random.uniform(-0.25, 0.25, 128)
    #     c2+=1
    elif str(user)!="nan" and user not in user_to_id:
        user_embeds[user]=user_embeds["dummy"]
        c2+=1        


In [ ]:
print("Known Users from training : " , c1)
print("New users : ",c2)

### Generating content features

In [ ]:
from sentence_transformers import SentenceTransformer, models
from torch import nn

word_embedding_model = models.Transformer("bert-base-uncased")
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
dense_model = models.Dense(
    in_features=pooling_model.get_sentence_embedding_dimension(),
    out_features=256,
    activation_function=nn.Tanh(),
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model, dense_model])
def process_sentences_in_batches(sentences, batch_size=1000):
    num_sentences = len(sentences)
    embeddings = []
    for i in range(0, num_sentences, batch_size):
        batch_sentences = sentences[i:i+batch_size]
        batch_embeddings = model.encode(batch_sentences)
        embeddings.extend(batch_embeddings)
        print(i)
    return torch.tensor(np.array(embeddings))


In [ ]:
# Sentence embedding - Bert model - Will take longer time (around 3-5 hours)
sentence_embeddings = process_sentences_in_batches(sentences,10000)
id_to_int = {id_: i for i, id_ in enumerate(ids)}
with open("sentence_embeds_bert.pkl","wb") as f:
     pickle.dump(sentence_embeddings,f)
        
with open("id_to_int.pkl","wb") as f:
     pickle.dump(id_to_int,f)

In [ ]:
def get_node_features_with_user_and_bert(nodes_dic,sentence_embeddings,id_to_int,id_to_user,user_embeds):
    num_nodes = len(nodes_dic)
    shape = (num_nodes,384)
    torch_tensor = torch.zeros(shape)
    # print(num_nodes)
    for i in nodes_dic:
        torch_tensor[nodes_dic[i]][:256] = sentence_embeddings[id_to_int[i]]
        
        user = id_to_user[i]
        
        if str(user)!="nan":
            torch_tensor[nodes_dic[i]][256:] = torch.tensor(user_embeds[user])
        else:
            #torch_tensor[nodes_dic[i]][256:] = torch.tensor(np.random.uniform(-0.25, 0.25, 128))
            torch_tensor[nodes_dic[i]][256:] = torch.tensor(user_embeds["dummy"])
            
    if torch.isnan(torch_tensor).any():
        print("SOME ISSUE")
        
    return torch_tensor

### Saving the processed dataset

In [ ]:
## Storing the dataset 
l=0
id_data_mapping = dict()

for name,group in grp:
    my_dic = dict()
    t=0
    my_dic[name[0]] = t
    t+=1
    
    for i,sent in zip(group["id"],group["body"]):
        if str(sent)!='nan':
           my_dic[i]=t
           t+=1

    group['id'] = group['id'].map(my_dic)
    group['parent_id'] = group['parent_id'].map(my_dic)
    
    edges_list =[]
    for id,p_id in zip(group['id'],group['parent_id']):
        edges_list.append([p_id,id])
        edges_list.append([id,p_id])
        
    edge_index = torch.tensor(edges_list)

    data = Data(x=get_node_features_with_user_and_bert(my_dic,sentence_embeddings,id_to_int,id_to_user,user_embeds),  ## Change this function according to data
                edge_index=edge_index.t().contiguous(),
                y=torch.tensor([int(labels_mapping[name[0]])]))

    if l%10000==0:
        print(l)
    if  torch.isnan(data.edge_index).any()==False:
        torch.save(data,"fakeddit_only_train/processed/data_"+str(l)+".pt")
        id_data_mapping[name]=l
        l+=1

In [ ]:
id_data_mapping_transformed = dict()
id_data_mapping_transformed={k[0]:v for k,v in id_data_mapping.items()}
id_data_mapping_transformed

In [ ]:
import json
with open("data_mapping_only_train.json","w") as f:
     p = json.dump(id_data_mapping_transformed,f)